# Shadow training — Kaggle GPU (parallel)

**One notebook trains one variant.** Set `VARIANT` in the first cell, then run top-to-bottom. Duplicate this notebook 3 times (one per variant) and run all three in parallel — Kaggle allows up to 4 concurrent GPU sessions per account. Total wall time ≈ 4 hours.

**Variants:**
- `rgb` — train on raw images (RGB baseline)
- `shadow_vanilla` — train on Shadow-edited images (paper's method)
- `shadow_noise` — train on Shadow-edited images with calibration noise injection (our extension)

**Setup checklist (one time):**
1. Create a Kaggle account, verify phone (required for GPU).
2. Push your shadow project to GitHub (it's ok if it's private — Kaggle can clone it with a token).
3. Set this notebook's *Settings → Accelerator → GPU P100*, and *Internet → On*.

In [ ]:
# ============== EDIT THIS ==============
VARIANT      = 'shadow_noise'            # rgb | shadow_vanilla | shadow_noise
SEED         = 0
NUM_EPOCHS   = 200                       # matched across all three variants for clean comparison
GITHUB_REPO  = 'https://github.com/gautvm/shadow-cross-embodiment.git'
# =======================================

assert VARIANT in {'rgb', 'shadow_vanilla', 'shadow_noise'}
print(f'will train {VARIANT} (seed {SEED}) for {NUM_EPOCHS} epochs')


## 1. Install dependencies (~5 min)

Pull robosuite v1.5.1, robomimic v0.5.0, mimicgen 1.0, and apply the three patches we documented in `CLAUDE.md`.

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d robosuite ] || git clone -q https://github.com/ARISE-Initiative/robosuite.git
[ -d robomimic ] || git clone -q https://github.com/ARISE-Initiative/robomimic.git
[ -d mimicgen  ] || git clone -q https://github.com/NVlabs/mimicgen.git
(cd robosuite && git checkout -q v1.5.1)

# --- patch 1: egl_probe is Linux-multi-GPU-only; harmless to keep on Kaggle P100 but easier to drop
sed -i 's/.*egl_probe.*$//' robomimic/setup.py
python - <<'PY'
p = 'robomimic/robomimic/envs/env_robosuite.py'
s = open(p).read()
if 'try:\n                    import egl_probe' not in s:
    s = s.replace(
        'import egl_probe\n                valid_gpu_devices = egl_probe.get_available_devices()',
        'try:\n                    import egl_probe\n                    valid_gpu_devices = egl_probe.get_available_devices()\n                except ImportError:\n                    valid_gpu_devices = []')
open(p, 'w').write(s)
PY

# --- patch 2: SingleArmEnv compat shim for mimicgen
cat > robosuite/robosuite/environments/manipulation/single_arm_env.py <<'PY'
from robosuite.environments.manipulation.manipulation_env import ManipulationEnv as SingleArmEnv  # noqa: F401
PY
echo 'patches applied'

In [ ]:
%%bash
set -e
cd /kaggle/working
pip install -q 'mujoco==3.2.7' cmake 'numpy<2.3' scipy seaborn
pip install -q -e ./robosuite
pip install -q -e ./robomimic
pip install -q -e ./mimicgen

# Kaggle's P100 GPU is Pascal (sm_60). Newer PyTorch (2.6+) dropped Pascal
# kernels — robomimic pulls torch>=2.0 which resolves to the latest, breaking
# CUDA on P100. Force-reinstall a Pascal-compatible build (2.4.1 + cu121).
pip install -q --force-reinstall --no-deps torch==2.4.1 torchvision==0.19.1 \
    --index-url https://download.pytorch.org/whl/cu121

# Root cause of every diffusers crash: Kaggle ships flax pre-installed,
# diffusers 0.11.1 detects it via is_flax_available() and tries to import
# Flax scheduler files that don't work with newer JAX/Python. Removing flax
# triggers diffusers' own dummy-stub fallback path. We use PyTorch only.
pip uninstall -y -q flax || true

python robosuite/robosuite/scripts/setup_macros.py >/dev/null 2>&1 || true
echo 'install done'


## 2. Clone the shadow project from GitHub

In [ ]:
%%bash -s "$GITHUB_REPO"
set -e
cd /kaggle/working
[ -d shadow ] || git clone -q "$1" shadow
cd shadow && git pull -q
ls

## 3. Build the dataset for this variant (~30–60 min)

RGB needs nothing — it uses the raw Mimicgen Stack dataset. Both Shadow variants run the IK+render+composite pipeline.

In [ ]:
%%bash -s "$VARIANT"
set -e
cd /kaggle/working/shadow
mkdir -p datasets/core datasets/shadow

REL=https://github.com/gautvm/shadow-cross-embodiment/releases/download/v0-data

case "$1" in
  rgb)
    # RGB trains on the raw Mimicgen Stack dataset.
    python /kaggle/working/mimicgen/mimicgen/scripts/download_datasets.py \
        --download_dir datasets --dataset_type core --tasks stack_d0
    ;;
  shadow_vanilla)
    # Pre-built Shadow-edited dataset (built locally to bypass Kaggle's slow CPU)
    wget -q --show-progress -O datasets/shadow/stack_d0_shadow.hdf5 "$REL/stack_d0_shadow.hdf5"
    ls -lh datasets/shadow/stack_d0_shadow.hdf5
    ;;
  shadow_noise)
    wget -q --show-progress -O datasets/shadow/stack_d0_shadow_noise.hdf5 "$REL/stack_d0_shadow_noise.hdf5"
    ls -lh datasets/shadow/stack_d0_shadow_noise.hdf5
    ;;
esac


## 4. Train (~2.5–3.5h on Kaggle P100)

In [ ]:
import subprocess
subprocess.run(
    ['python', 'train.py',
     '--variant', VARIANT,
     '--seed', str(SEED),
     '--epochs', str(NUM_EPOCHS)],
    cwd='/kaggle/working/shadow',
    check=True,
)

## 5. Save the checkpoint to Kaggle's output (so you can download it)

In [ ]:
%%bash -s "$VARIANT" "$SEED"
set -e
# Robomimic writes runs to <its install dir>/results/runs/<exp_name>/<timestamp>/.
# We disabled in-training rollouts (they leak GL ctx and crash long runs), so
# only `last.pth` is produced — saved every epoch, overwritten in place.
RUN_DIR=$(ls -td /kaggle/working/robomimic/robomimic/results/runs/${1}_seed${2}/* 2>/dev/null | head -1)
if [ -z "$RUN_DIR" ]; then
    RUN_DIR=$(ls -td /kaggle/working/shadow/results/runs/${1}_seed${2}/* 2>/dev/null | head -1)
fi
CKPT="$RUN_DIR/last.pth"
echo "saving $CKPT"
test -f "$CKPT" || { echo "ERROR: no last.pth at $CKPT"; find /kaggle/working -name 'last.pth' 2>/dev/null; exit 1; }
mkdir -p /kaggle/working/checkpoints
cp "$CKPT" "/kaggle/working/checkpoints/${1}_seed${2}.pth"
ls -lh /kaggle/working/checkpoints/